# Stage 17B: uncovered short-prefix selector screen

Stage 17Aで公開OOFを使えなかった短prefix cutだけに、V599/SP45のlikelihood particle-filter信号を再生します。CPU-onlyのscreenで、Kaggle提出は行いません。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess
REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
drive_root = Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir = drive_root / 'artifacts'
data_dir = drive_root / 'data'
if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)
assert (data_dir / 'train').is_dir()
artifact_dir.mkdir(parents=True, exist_ok=True)
def run_checked(command):
    result = subprocess.run(command, cwd=repo_dir, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout, flush=True)
    if result.returncode != 0:
        print(result.stderr, flush=True)
        raise RuntimeError(f'Command failed with exit code {result.returncode}: {command}')
    return result
subprocess.run(['git', '-C', str(repo_dir), 'rev-parse', '--short', 'HEAD'], check=True)

## 固定入力

Stage 16B v003と、通過済みStage 17A v002を使います。

In [ ]:
stage16b_run = artifact_dir / 'stage16b_testlike_validation_full_v003'
stage17a_run = artifact_dir / 'stage17_public_replay_full_v002'
assert (stage16b_run / 'pseudo_test_manifest.parquet').is_file(), 'Missing Stage 16B v003'
assert (stage17a_run / 'cut_report.parquet').is_file(), 'Missing Stage 17A v002'
print('Stage 16B:', stage16b_run)
print('Stage 17A:', stage17a_run)

## 8坑井smoke

初回だけNumba compileに時間がかかります。suffix TVTは予測関数へ渡さず、hidden-target invarianceを維持します。

In [ ]:
SMOKE_ID = 'stage17b_selector_replay_smoke_v001'
smoke_dir = artifact_dir / SMOKE_ID
if not (smoke_dir / 'summary.json').is_file():
    run_checked([
        'uv', 'run', 'rogii-selector-replay',
        '--config', 'configs/experiment/stage17b_selector_replay.yaml',
        '--stage16b-run', str(stage16b_run), '--stage17a-run', str(stage17a_run),
        '--data-dir', str(data_dir), '--artifact-dir', str(artifact_dir),
        '--run-id', SMOKE_ID, '--limit-wells', '8',
    ])
smoke = json.loads((smoke_dir / 'summary.json').read_text())
assert smoke['stage17b_complete'] and smoke['gates']['hidden_target_invariance']
smoke

## 全uncovered cut screen

8 seeds × 96 particles、最大512 tracking stepsでscreenします。full V599より軽量で、beamはまだ適用しません。

In [ ]:
RUN_ID = 'stage17b_selector_replay_full_v001'
run_dir = artifact_dir / RUN_ID
if not (run_dir / 'summary.json').is_file():
    run_checked([
        'uv', 'run', 'rogii-selector-replay',
        '--config', 'configs/experiment/stage17b_selector_replay.yaml',
        '--stage16b-run', str(stage16b_run), '--stage17a-run', str(stage17a_run),
        '--data-dir', str(data_dir), '--artifact-dir', str(artifact_dir), '--run-id', RUN_ID,
    ])
else:
    print('Reusing completed run:', run_dir)
summary = json.loads((run_dir / 'summary.json').read_text())
{
    'stage17b_complete': summary['stage17b_complete'],
    'promoted_to_full_selector_validation': summary['promoted_to_full_selector_validation'],
    'n_uncovered_cuts': summary['n_uncovered_cuts'],
    'selector_profile': summary['selector_profile'],
    'primary': summary['role_report']['primary'],
    'diagnostic': summary['role_report']['diagnostic'],
    'improved_folds': summary['improved_folds'],
    'fold_report': summary['fold_report'],
    'gates': summary['gates'],
    'approximation_note': summary['approximation_note'],
    'next_step': summary['next_step'],
}

最後の辞書を共有してください。`promoted_to_full_selector_validation`がfalseなら、PF/beamへ計算量を追加せずStage 18 retrievalへ移ります。